In [2]:
!pip install -q pypdf sentence-transformers chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [5]:
from google.colab import userdata, files
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

In [1]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")

print("OpenAI API key loaded!")

OpenAI API key loaded!


In [3]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_NAME = "gpt-5.6-luna"

print("OpenAI client ready!")

OpenAI client ready!


In [6]:
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print("PDF uploaded:", pdf_filename)

Saving 33_Tutorial 2 Chain.docx.pdf to 33_Tutorial 2 Chain.docx.pdf
PDF uploaded: 33_Tutorial 2 Chain.docx.pdf


In [7]:
reader = PdfReader(pdf_filename)

pdf_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text + "\n"

print("PDF text extracted!")
print("Characters:", len(pdf_text))

PDF text extracted!
Characters: 8856


In [8]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(pdf_text)

print("Total chunks:", len(chunks))

Total chunks: 12


In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [10]:
embeddings = embedder.encode(chunks).tolist()

print("Embeddings created!")
print("Total embeddings:", len(embeddings))

Embeddings created!
Total embeddings: 12


In [11]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="pdf_chatbot"
)

print("ChromaDB ready!")

ChromaDB ready!


In [12]:
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print("PDF chunks stored!")

PDF chunks stored!


In [13]:
def retrieve_pdf_context(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results["documents"][0]

In [14]:
def ask_pdf(query):
    context = retrieve_pdf_context(query, 3)
    context_str = "\n".join(context)

    prompt = f"""
Answer using ONLY the PDF context.

If the answer is not in the PDF context, say:
"I cannot find the answer in the provided PDF."

PDF Context:
{context_str}

Question:
{query}
"""

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )

    return response.output_text, context

In [16]:
question = input("Ask a question about your PDF: ")

answer, context = ask_pdf(question)

print("\n" + "=" * 50)
print("ANSWER")
print("=" * 50)

print(answer)

Ask a question about your PDF: Tell me  Packaging and Sealing Details 

ANSWER
Each digital device is placed in a separate evidence package.

**Evidence 1 – Company Laptop**
- Evidence ID: DF-2026-002-L
- Description: Company Laptop
- Date Seized: 25-07-2026
- Collected By: Anand Sharma
- Packaging: Tamper-evident evidence bag
- Seal Condition: Intact

**Evidence 2 – External HDD**
- Evidence ID: DF-2026-002-H
- Description: External Hard Disk Drive
- Date Seized: 25-07-2026
- Collected By: Anand Sharma
- Packaging: Tamper-evident evidence bag
- Seal Condition: Intact

**Evidence 3 – USB Drive**
- Evidence ID: DF-2026-002-U
- Description: USB Flash Drive
- Date Seized: 25-07-2026
- Collected By: Anand Sharma
- Packaging: Tamper-evident evidence bag
- Seal Condition: Intact

Evidence packages were sealed using tamper-evident evidence tape. The seals were inspected and found intact at the time of transfer.


In [ ]:
print("=" * 50)
print("       PDF CHATBOT READY!")
print("=" * 50)

while True:

    question = input("\nAsk a question: ")

    if question.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not question.strip():
        continue

    answer, context = ask_pdf(question)

    print("\n" + "-" * 50)
    print("ANSWER")
    print("-" * 50)

    print(answer)

       PDF CHATBOT READY!

Ask a question: what is the pdf about

--------------------------------------------------
ANSWER
--------------------------------------------------
The PDF is about a case study involving the suspected theft of confidential employee information by copying it from a company laptop to an external HDD and USB drive. It covers the seizure and preservation of digital evidence, chain of custody, evidence presentation, and forensic examination for employee information files, payroll documents, HR records, recently accessed files, and file-copy activity.

Ask a question: give me Forensic Examination Entry details in table form

--------------------------------------------------
ANSWER
--------------------------------------------------
| Forensic Examination Entry | Details |
|---|---|
| Evidence ID | Verified |
| Device description | Verified |
| Packaging | Verified |
| Seal condition | Verified |
| Serial number | Verified |
| Device identification information | Ve